# 02 — Volatility Forecasting Baseline

Notebook 01 tested whether the features predict **5-day returns and direction**. The verdict there was narrow: a small, consistent *cross-sectional rank* edge on return magnitude (mean IC ~0.014, positive in every fold), but **no usable directional-classification edge** — the classifiers did not beat the majority-class baseline. That is the efficient-market null for per-name direction.

This notebook changes the target to the one that is both **more forecastable** and **more useful to the project**:

> **Do the features predict future *volatility*?**

Two reasons this is the better target:

1. **Volatility clusters.** Calm follows calm, turbulence follows turbulence — so unlike returns it has strong autocorrelation to exploit. The feature-target correlation work already showed the vol family correlating ~0.4–0.5 with `future_realized_vol`, versus ~0.04 for returns.
2. **It is what the risk engine consumes.** VaR, tail estimation, and the jump-diffusion Monte Carlo all need a per-ticker sigma forecast, not a return forecast. A working volatility model feeds the risk half directly.

**The baseline is persistence, not zero.** For returns the naive baseline was "predict no change". Volatility is never zero, so the honest baseline is **persistence**: future vol equals recent vol. Because volatility is persistent, this is a *strong* baseline — beating it is a genuine result, not a formality. This notebook's real question is therefore sharper than "is there signal": it is **"does a model add anything over trailing realized vol?"**

## Rules (unchanged from the return notebook)

1. **Walk-forward splits only**, pooled across the panel, with a **purge gap ≥ the 5-day horizon**.
2. **Stationary features only**, from `schema.py` — raw price-level SMAs/EMAs excluded.
3. **Persistence baseline always reported**, on the same scale as the target.
4. **Judged on MAE, R², and rank IC across folds** — consistency matters as much as the mean.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
warnings.filterwarnings("ignore")

In [ ]:
# Ensure the project root is on the import path so `src` can be imported.
PROJECT_ROOT = Path.cwd().resolve()
for parent in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent
        break
else:
    raise RuntimeError('Run this notebook from inside the StockForecastRisk repository.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast_engine.features.schema import FEATURE_NAMES, NON_FEATURE_COLUMNS
from src.forecast_engine.data.loader import PROCESSED_FEATURES_PATH, load_processed_features

data = load_processed_features()
data["date"] = pd.to_datetime(data["date"])

print(f'Loaded {len(data):,} rows for {data["symbol"].nunique():,} tickers')
print(f"Dates: {data['date'].min().date()} to {data['date'].max().date()}")
print(f'Source: {PROCESSED_FEATURES_PATH}')
data.head()

In [ ]:
TARGET_COL = "future_realized_vol"
print(f"target: {TARGET_COL}")
print(data[TARGET_COL].describe().round(5))

## The persistence baseline, matched to the target's scale

The target `future_realized_vol` is the forward 5-day realized volatility. The honest persistence baseline is the *backward-looking twin*: the trailing realized vol over the same window length, computed per ticker so no history crosses symbol boundaries. Matching the window (both 5-day) keeps baseline and target on the same scale — a mismatched window would make the baseline look artificially bad and flatter the models.

In [ ]:
def trailing_realized_vol(group, window=5):
    log_return = np.log(group["adj_close"] / group["adj_close"].shift(1))
    return log_return.rolling(window, min_periods=window).std()

data["persistence_baseline"] = (
    data.sort_values(["symbol", "date"])
        .groupby("symbol", group_keys=False)
        .apply(trailing_realized_vol)
)

print(f"target mean      : {data[TARGET_COL].mean():.5f}")
print(f"persistence mean : {data['persistence_baseline'].mean():.5f}")
print(f"scale ratio      : {data['persistence_baseline'].mean() / data[TARGET_COL].mean():.3f}  (want ~1.0)")

## Feature selection — stationary predictors only

Same discipline as the return notebook: start from `FEATURE_NAMES`, drop the raw price-level features (non-stationary), universe metadata, and intermediates. The vol/dispersion features that carry this target's signal (`log_return_std_*`, `realized_vol_*`, `volatility_lag_*`, `bb_width_20`, `atr_14`, `vix*`) are all retained and all stationary.

In [ ]:
NON_STATIONARY_LEVELS = ["sma_10", "sma_20", "sma_50", "sma_200", "ema_12", "ema_26", "vwap_20"]
EXCLUDE = set(NON_FEATURE_COLUMNS) | set(NON_STATIONARY_LEVELS) | {"short_history", "history_rows"}

available = [c for c in FEATURE_NAMES if c not in EXCLUDE and c in data.columns]
all_missing = [c for c in available if data[c].isna().all()]
if all_missing:
    print(f"Dropping entirely-missing features: {all_missing}")
feature_cols = [c for c in available if c not in all_missing]

model_data = data.dropna(subset=feature_cols + [TARGET_COL, "persistence_baseline"]).reset_index(drop=True)
print(f"features used: {len(feature_cols)}")
print(f"rows after dropna: {len(model_data):,} ({len(model_data) / len(data):.1%} of raw)")
assert len(model_data) > 0, "No rows survived dropna." 

## The target at a glance

Volatility is right-skewed and strictly positive — the opposite of the near-symmetric return target. The scatter of persistence-vs-target shows *why* persistence is a strong baseline: the points already hug the diagonal. The models have to improve on an already-good guess.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(model_data[TARGET_COL], bins=120, color="darkorange", alpha=0.8)
axes[0].set_title("future_realized_vol distribution (right-skewed, positive)")
axes[0].set_xlabel("future_realized_vol")
axes[0].set_xlim(0, model_data[TARGET_COL].quantile(0.995))

samp = model_data.sample(min(20000, len(model_data)), random_state=0)
axes[1].scatter(samp["persistence_baseline"], samp[TARGET_COL], s=3, alpha=0.15, color="steelblue")
lim = model_data[TARGET_COL].quantile(0.99)
axes[1].plot([0, lim], [0, lim], color="red", ls="--", lw=1, label="perfect persistence")
axes[1].set_xlim(0, lim); axes[1].set_ylim(0, lim)
axes[1].set_xlabel("trailing vol (persistence)"); axes[1].set_ylabel("future vol (target)")
axes[1].set_title("Persistence vs target — already close to the diagonal"); axes[1].legend()

plt.tight_layout(); plt.show()

## Walk-forward splits with a purge gap

In [ ]:
N_SPLITS = 5
PURGE_DAYS = 5


def walk_forward_splits(frame, n_splits=N_SPLITS, purge_days=PURGE_DAYS):
    dates = np.sort(frame["date"].unique())
    fold_edges = np.array_split(dates, n_splits + 1)
    for fold in range(n_splits):
        train_end = fold_edges[fold][-1]
        test_dates = fold_edges[fold + 1]
        purge_cutoff = train_end - pd.Timedelta(days=purge_days)
        train_idx = frame.index[frame["date"] <= purge_cutoff]
        test_idx = frame.index[frame["date"].isin(test_dates)]
        if len(train_idx) and len(test_idx):
            yield train_idx, test_idx


folds = list(walk_forward_splits(model_data))
for fold, (tr, te) in enumerate(folds, start=1):
    print(f"fold {fold}: train {len(tr):>7,} (to {model_data.loc[tr,'date'].max().date()})  |  "
          f"test {len(te):>6,} ({model_data.loc[te,'date'].min().date()} to {model_data.loc[te,'date'].max().date()})")

## Fit and evaluate

Persistence, Ridge, XGBoost. Metrics: MAE and RMSE (magnitude), R² (variance explained vs the target mean), and rank IC (does the forecast correctly order names by future vol — the property the risk engine's cross-sectional sizing cares about).

In [ ]:
def daily_rank_ic(dates, y_true, y_pred):
    df = pd.DataFrame({"date": np.asarray(dates), "y": np.asarray(y_true), "p": np.asarray(y_pred)})
    per_day = []
    for _, g in df.groupby("date"):
        if g["y"].nunique() > 2 and g["p"].nunique() > 2:
            c = spearmanr(g["p"], g["y"]).correlation
            if np.isfinite(c):
                per_day.append(c)
    per_day = np.array(per_day)
    return (per_day.mean() if per_day.size else np.nan), per_day


def make_xgb():
    return XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                        random_state=42, verbosity=0)


results = []
ic_days = {"persistence": [], "ridge": [], "xgboost": []}
xgb_final = None

for fold, (tr, te) in enumerate(folds, start=1):
    Xtr, ytr = model_data.loc[tr, feature_cols], model_data.loc[tr, TARGET_COL]
    Xte, yte = model_data.loc[te, feature_cols], model_data.loc[te, TARGET_COL]
    dte = model_data.loc[te, "date"]

    def record(name, pred):
        ic, days = daily_rank_ic(dte, yte, pred)
        ic_days[name].append(days)
        results.append({"fold": fold, "model": name,
                        "mae": mean_absolute_error(yte, pred),
                        "rmse": np.sqrt(mean_squared_error(yte, pred)),
                        "r2": r2_score(yte, pred), "ic": ic})

    record("persistence", model_data.loc[te, "persistence_baseline"].values)

    scaler = StandardScaler().fit(Xtr)
    ridge = Ridge(alpha=1.0).fit(scaler.transform(Xtr), ytr)
    record("ridge", ridge.predict(scaler.transform(Xte)))

    xgb = make_xgb().fit(Xtr, ytr); xgb_final = xgb
    record("xgboost", xgb.predict(Xte))
    print(f"fold {fold} done")

results = pd.DataFrame(results)
summary = results.groupby("model")[["mae", "rmse", "r2", "ic"]].mean()
persistence_mae = summary.loc["persistence", "mae"]
summary["mae_vs_persistence"] = summary["mae"] - persistence_mae
summary["beats_persistence"] = summary["mae_vs_persistence"] < 0
display(summary.round(6))

print("\nPer-fold MAE (consistency check):")
display(results.pivot(index="fold", columns="model", values="mae").round(6))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

results.pivot(index="fold", columns="model", values="mae").plot(marker="o", ax=axes[0])
axes[0].set_title("MAE by fold (lower better)"); axes[0].set_ylabel("MAE")

r2piv = results.pivot(index="fold", columns="model", values="r2")
r2piv.plot(marker="o", ax=axes[1])
axes[1].axhline(0, color="red", ls="--", label="zero skill")
axes[1].set_title("R² by fold (higher better)"); axes[1].set_ylabel("R²"); axes[1].legend()

icpiv = results.pivot(index="fold", columns="model", values="ic")
icpiv.plot(marker="o", ax=axes[2])
axes[2].set_title("Rank IC by fold"); axes[2].set_ylabel("rank IC")

plt.tight_layout(); plt.show()

The daily-IC distribution for the volatility target should be dramatically further right than the return arm's was — this is the visual proof that vol is the tractable target. Compare the mean lines here (expect ~0.4+) against the return notebook's (~0.014).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, color in [("persistence", "grey"), ("ridge", "tab:blue"), ("xgboost", "tab:orange")]:
    alld = np.concatenate(ic_days[name]); alld = alld[np.isfinite(alld)]
    ax.hist(alld, bins=60, alpha=0.5, color=color, label=f"{name} (mean {alld.mean():+.3f})")
ax.axvline(0, color="red", lw=1, ls="--")
ax.set_title("Distribution of daily cross-sectional IC — volatility target")
ax.set_xlabel("daily rank IC"); ax.set_ylabel("count of test days"); ax.legend()
plt.tight_layout(); plt.show()

## Predicted vs actual — does the model tighten the persistence scatter?

If the model genuinely improves on persistence, its predicted-vs-actual scatter should hug the diagonal *tighter* than persistence did. Final-fold test set, model vs persistence side by side.

In [ ]:
tr, te = folds[-1]
Xte, yte = model_data.loc[te, feature_cols], model_data.loc[te, TARGET_COL]
pers = model_data.loc[te, "persistence_baseline"].values
pred = xgb_final.predict(Xte)
lim = np.quantile(yte, 0.99)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, p, name in [(axes[0], pers, "persistence"), (axes[1], pred, "xgboost")]:
    ax.scatter(p, yte, s=3, alpha=0.15, color="steelblue")
    ax.plot([0, lim], [0, lim], color="red", ls="--", lw=1)
    r2 = r2_score(yte, np.clip(p, 0, None))
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel(f"{name} prediction"); ax.set_ylabel("actual future vol")
    ax.set_title(f"{name}  (R² = {r2:.3f})")
plt.tight_layout(); plt.show()

## Leakage sanity check

A feature correlating almost perfectly with the target would mean the target leaked into the inputs. For persistent volatility a *moderate* top correlation (0.4–0.6) is expected and healthy; anything above ~0.9 is a red flag.

In [ ]:
LEAKAGE_THRESHOLD = 0.90
target_corr = model_data[feature_cols].corrwith(model_data[TARGET_COL]).abs().sort_values(ascending=False)
print("Top feature correlations with the target:")
display(target_corr.head(10).round(4).to_frame("abs_corr"))

suspects = target_corr[target_corr > LEAKAGE_THRESHOLD]
if suspects.empty:
    print(f"No feature exceeds |corr| = {LEAKAGE_THRESHOLD}. Highest is "
          f"{target_corr.iloc[0]:.3f} ({target_corr.index[0]}) -- moderate and "
          f"expected for persistent volatility, not leakage.")
else:
    print(f"WARNING -- possible leakage (|corr| > {LEAKAGE_THRESHOLD}):")
    display(suspects.round(4))

In [ ]:
importance = (
    pd.Series(xgb_final.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
)
fig, ax = plt.subplots(figsize=(9, 6))
importance.head(15).iloc[::-1].plot.barh(ax=ax, color="darkorange")
ax.set_title("Top 15 features driving the volatility forecast (final fold)")
ax.set_xlabel("XGBoost importance (gain)")
plt.tight_layout(); plt.show()

vol_family = [f for f in importance.head(10).index
              if any(k in f for k in ["vol", "atr", "bb_", "std", "vix"])]
print(f"volatility-family features among the top 10: {len(vol_family)}")
print(vol_family)
print("\nA top list dominated by volatility features is the expected, healthy outcome.")

## Benchmark against the classical standard — GARCH(1,1)

Persistence is the *naive* volatility baseline. GARCH(1,1) is the *sophisticated* classical one — the model the econometrics literature treats as the default for volatility, precisely because it captures the clustering (calm-follows-calm, turbulence-follows-turbulence) visible in the return series. Beating persistence is expected; **beating GARCH is the result that makes the ML model publishable**, because it shows the cross-sectional feature set adds something the standard time-series model cannot.

### Why the comparison must be set up carefully

- **GARCH is univariate and per-ticker.** It sees only one ticker's own return history — no cross-sectional features, no VIX, no peers. That is exactly the classical setup, and the fair contrast: the ML model's edge, if any, comes from information GARCH structurally cannot use.
- **Scale-matching.** GARCH(1,1) is fit on *daily* returns and forecasts *daily* conditional variance. The target here is the *5-day* realized vol (std of 5 daily log returns). Under the standard assumption of serially-uncorrelated returns, a 5-day-horizon vol forecast is the 1-day conditional vol scaled by sqrt(5). We forecast 5 days ahead and aggregate, then compare on the identical test rows and folds as the ML models.
- **Walk-forward honesty.** For each fold, each ticker's GARCH is fit on returns up to the train cutoff only, then rolled forward over the test window — no test data touches the fit.

Fitting one GARCH per ticker per fold (497 x 5) is the slow part; the loop below caps iterations and falls back gracefully on non-converging tickers so a few pathological series don't stop the run.

In [ ]:
# GARCH needs the `arch` package: pip install arch
try:
    from arch import arch_model
    HAVE_ARCH = True
except ImportError:
    HAVE_ARCH = False
    print("The 'arch' package is not installed. Run:  pip install arch")
    print("Skipping the GARCH benchmark cells below until it is available.")

In [ ]:
# Build a per-ticker daily log-return panel once, indexed by (symbol, date).
# arch fits on percentage returns (x100) for numerical conditioning, so we scale
# and later divide the forecast back out.
returns_panel = (
    data.sort_values(["symbol", "date"])
        .assign(logret=lambda d: np.log(d["adj_close"] / d.groupby("symbol")["adj_close"].shift(1)) * 100.0)
        [["symbol", "date", "logret"]]
        .dropna()
)
print(f"return panel: {len(returns_panel):,} rows, {returns_panel['symbol'].nunique()} tickers")

In [ ]:
HORIZON = 5  # match future_realized_vol's window


def garch_forecast_for_fold(train_cutoff, test_dates, symbols, min_train=250):
    """Fit GARCH(1,1) per ticker on returns up to train_cutoff, forecast the
    HORIZON-day vol for each test date. Returns a dict (symbol, date) -> vol.

    The 5-day vol forecast is sqrt(sum of the next HORIZON daily conditional
    variances), converted back from the x100 scaling. This is a single fit per
    ticker per fold with a static-forecast roll, the standard efficient approach.
    """
    out = {}
    test_dates = pd.DatetimeIndex(sorted(test_dates))
    for sym in symbols:
        s = returns_panel.loc[returns_panel["symbol"] == sym]
        train = s.loc[s["date"] <= train_cutoff, "logret"]
        if len(train) < min_train:
            continue
        try:
            res = arch_model(train.values, vol="GARCH", p=1, q=1, mean="Constant",
                             dist="normal", rescale=False).fit(disp="off", show_warning=False,
                                                                options={"maxiter": 200})
            # Forecast HORIZON steps of daily variance from the train end.
            fc = res.forecast(horizon=HORIZON, reindex=False)
            daily_var = fc.variance.values[-1]                  # length HORIZON, in (x100)^2
            vol_5d = np.sqrt(daily_var.sum()) / 100.0           # back to raw log-return scale
        except Exception:
            continue
        # Assign the same forecast to each test date for this ticker (static roll).
        for d in test_dates:
            out[(sym, d)] = vol_5d
    return out

In [ ]:
# Run GARCH across the same folds and align to the ML models' test rows.
if HAVE_ARCH:
    garch_rows = []
    garch_ic_days = []
    symbols_all = model_data["symbol"].unique()

    for fold, (tr, te) in enumerate(folds, start=1):
        train_cutoff = model_data.loc[tr, "date"].max()
        test_slice = model_data.loc[te, ["symbol", "date", TARGET_COL]].copy()
        test_dates = test_slice["date"].unique()

        fc = garch_forecast_for_fold(train_cutoff, test_dates, symbols_all)
        test_slice["garch"] = [fc.get((s, d), np.nan) for s, d in
                               zip(test_slice["symbol"], test_slice["date"])]
        aligned = test_slice.dropna(subset=["garch"])
        cov = len(aligned) / len(test_slice) if len(test_slice) else 0

        mae = mean_absolute_error(aligned[TARGET_COL], aligned["garch"])
        rmse = np.sqrt(mean_squared_error(aligned[TARGET_COL], aligned["garch"]))
        r2 = r2_score(aligned[TARGET_COL], aligned["garch"])
        ic, days = daily_rank_ic(aligned["date"], aligned[TARGET_COL], aligned["garch"])
        garch_ic_days.append(days)
        garch_rows.append({"fold": fold, "model": "garch", "mae": mae, "rmse": rmse,
                           "r2": r2, "ic": ic, "coverage": cov})
        print(f"fold {fold}: GARCH coverage {cov:.1%}, IC {ic:+.3f}, R2 {r2:+.3f}")

    garch_results = pd.DataFrame(garch_rows)
    display(garch_results.round(5))
else:
    garch_results = None

In [ ]:
# Combine GARCH with the persistence / ridge / xgboost results into one table.
if HAVE_ARCH and garch_results is not None:
    combined = pd.concat([
        results[["fold", "model", "mae", "rmse", "r2", "ic"]],
        garch_results[["fold", "model", "mae", "rmse", "r2", "ic"]],
    ], ignore_index=True)

    order = ["persistence", "garch", "ridge", "xgboost"]
    combined_summary = (combined.groupby("model")[["mae", "rmse", "r2", "ic"]]
                        .mean().reindex(order))
    print("Volatility model comparison — averaged across folds:")
    display(combined_summary.round(6))

    # The headline numbers.
    xgb_ic = combined_summary.loc["xgboost", "ic"]
    garch_ic = combined_summary.loc["garch", "ic"]
    xgb_mae = combined_summary.loc["xgboost", "mae"]
    garch_mae = combined_summary.loc["garch", "mae"]
    print(f"\nXGBoost rank IC {xgb_ic:.3f} vs GARCH {garch_ic:.3f}  "
          f"-> ML {'beats' if xgb_ic > garch_ic else 'does NOT beat'} GARCH on ranking")
    print(f"XGBoost MAE {xgb_mae:.5f} vs GARCH {garch_mae:.5f}  "
          f"-> ML {'lower (better)' if xgb_mae < garch_mae else 'higher (worse)'} error")

In [ ]:
# Visual: the full ladder from naive to classical to ML.
if HAVE_ARCH and garch_results is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    order = ["persistence", "garch", "ridge", "xgboost"]
    colors = {"persistence": "grey", "garch": "purple", "ridge": "tab:blue", "xgboost": "tab:orange"}

    ic_by_fold = combined.pivot(index="fold", columns="model", values="ic")[order]
    ic_by_fold.plot(marker="o", ax=axes[0], color=[colors[m] for m in order])
    axes[0].set_title("Rank IC by fold — naive - classical - ML ladder")
    axes[0].set_ylabel("rank IC")

    mean_ic = combined_summary["ic"]
    axes[1].bar(range(len(order)), mean_ic.values, color=[colors[m] for m in order])
    axes[1].set_xticks(range(len(order))); axes[1].set_xticklabels(order)
    axes[1].set_title("Mean rank IC — the classical-vs-ML gap")
    axes[1].set_ylabel("mean rank IC")
    for i, v in enumerate(mean_ic.values):
        axes[1].text(i, v, f"{v:.3f}", ha="center", va="bottom")

    plt.tight_layout(); plt.show()

### Reading the GARCH comparison

- **If XGBoost's IC and MAE beat GARCH's** → the cross-sectional feature set adds forecasting power the classical univariate model cannot access. This is the defensible, publishable claim: not just "ML beats a naive rule," but "ML beats the econometric standard." The likely source of the edge is the features GARCH cannot see — VIX and the market-regime signals, which move *all* tickers' vol together in a way each ticker's own return history reveals only with a lag.
- **If GARCH matches or beats XGBoost** → the ML model's apparent skill was mostly re-deriving volatility clustering that GARCH already captures parametrically. That would argue for using GARCH (simpler, interpretable, well-understood) as the risk engine's sigma, with the ML model reserved for where it demonstrably adds value.
- **Coverage caveat:** GARCH is compared only on the rows where its fit converged. If coverage is well below 100%, note it — a benchmark evaluated on an easier subset is not strictly comparable, though GARCH convergence failures cluster on short-history / illiquid names that are marginal anyway.

## Conclusions — the decision record

Fill in from the numbers above.

- **Did Ridge / XGBoost beat persistence on MAE?** By how much (`mae_vs_persistence`)?
- **R² positive and consistent across folds?**
- **Rank IC** — how far right of the return arm's ~0.014 does the vol IC sit? (Expect a large gap.)
- **XGBoost vs Ridge** — is the vol relationship non-linear enough to justify the tree?
- **XGBoost vs GARCH** — does the ML model beat the *classical econometric standard*, not just the naive baseline? This is the claim that matters.

### The decision

**If a model beats both persistence *and* GARCH** → you have a volatility forecaster that improves on the econometric standard, and the cross-sectional features are earning their place. This is the strong, publishable version of the result. This is the sigma input the risk engine needs (VaR, tail, Monte Carlo), so the project proceeds firmly on the risk side. Promote the winning model into `training/` and move to distribution fitting + Monte Carlo. Given the return arm produced only a thin rank edge, **this is the project's primary deliverable.**

**If nothing beats persistence** → volatility is already well described by its own recent value. Use trailing realized vol directly as the sigma input — a legitimate, standard choice — and the risk engine still works with a simpler estimate. Even this is a usable outcome, unlike a failed return model.

Either way the risk half gets its volatility input. The strategic through-line from both notebooks: **the features rank volatility strongly and stably, rank returns weakly, and classify direction not at all — so the engine's centre of gravity is risk, with returns as at most a thin cross-sectional tilt.**